# Google Colab Remote Execution Environment

Use this notebook connected to a Colab T4 GPU (or better) when running computationally expensive models and tuning. This will drastically reduce run time and the strain on your own system, but requires remotely connecting to Google Colab and setting up the file structure, then copying files over and back.

## Set up Colab environment

Mount Google Drive, define the git ref to check out in Colab, and define the Drive location where completed tuning artifacts should be copied.

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive', force_remount=True)

REPO_URL = 'https://github.com/Jdrice72901/DSBA-6211-Stack-overflow-Survey-Analysis.git'
GIT_REF = 'sat_testing'
REPO_DIR = Path('/content/Stack-Overflow-Survey-Analysis')
DRIVE_ROOT = Path('/content/drive/MyDrive/stack-overflow-survey-analysis')
STUDY_NAME = 'sat_lightgbm_optuna_weighted'
LOCAL_OUTPUT_DIR = REPO_DIR / 'data' / 'outputs' / STUDY_NAME
DRIVE_OUTPUT_DIR = DRIVE_ROOT / 'outputs' / STUDY_NAME

Mounted at /content/drive


Clone repo if it doesn't exist and set it as the working directory

In [ ]:
# If you want to pull from main you don't need "-b {GIT_REF} --single-branch"

if not REPO_DIR.exists():
    !git clone -b {GIT_REF} --single-branch {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

Cloning into '/content/Stack-Overflow-Survey-Analysis'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 195 (delta 29), reused 33 (delta 15), pack-reused 147 (from 1)
Receiving objects: 100% (195/195), 94.87 MiB | 41.02 MiB/s, done.
Resolving deltas: 100% (81/81), done.
Filtering content: 100% (15/15), 1.25 GiB | 27.25 MiB/s, done.
/content/Stack-Overflow-Survey-Analysis


Install dependencies. This can't be done through the requirements.txt since Colab uses Python 3.12.13 and the repo is set for 3.13.12, so the dependencies are different. LightGBM also has optional optimization for CUDA cores if using an NVidua GPU.

If using the CUDA optimization, make sure you run this in a high RAM instance since those have more vCPUs and obviously more RAM since using CUDA requires build lightgbm from the source.

In [ ]:
USE_CUDA_LIGHTGBM = False

%pip install --upgrade pip setuptools wheel
%pip install --no-cache-dir \
    numpy \
    pandas \
    pyarrow \
    scikit-learn \
    optuna \
    catboost==1.2.10 \
    statsmodels

if USE_CUDA_LIGHTGBM:
    %pip install --no-cache-dir cmake ninja
    %pip uninstall -y lightgbm
    %pip install -v --no-cache-dir --force-reinstall --no-binary lightgbm lightgbm==4.6.0 --config-settings=cmake.define.USE_CUDA=ON
    LIGHTGBM_GPU_BACKEND = 'cuda'
else:
    %pip install --no-cache-dir lightgbm==4.6.0
    LIGHTGBM_GPU_BACKEND = 'gpu'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 98.2 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 127.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 369.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.1/615.1 kB 1.1 GB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [optuna]2m6/7 [optuna]]]y]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 18.3 MB/s  0:00:006m-:--:--


## Model Runs

Make sure the model is run on GPU with device-id set to 0 to select the correct one. Add any other needed arguments and set a script to copy outputs back over to the cloud Drive when done. This can take a long time to run even on power GPUs, 3+ hours for the most comprehensive modeling runs. If you run it overnight without copying the files over (or if there's a bug in how you set the copy script) you run the risk of the remote Colab Server idling too long and deleting itself along with your outputs.

### Satisfaction LightGBM Optuna Tuning

Run the study and immediately copy the actual study output directory to Google Drive when the script exits.

In [4]:
LOCAL_OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Local study dir: {LOCAL_OUTPUT_DIR}")
print(f"Drive copy dir: {DRIVE_OUTPUT_DIR}")

Local study dir: /content/Stack-Overflow-Survey-Analysis/data/outputs/sat_lightgbm_optuna_weighted
Drive copy dir: /content/drive/MyDrive/stack-overflow-survey-analysis/outputs/sat_lightgbm_optuna_weighted


In [7]:
!python -m src.tune_satisfaction_lightgbm --study-name sat_lightgbm_optuna_weighted --objective-metric weighted_f1 --spec core_with_comp --n-trials 100 --seed-list 42,52 --final-seed-list 42,52,62

for src_path in sorted(LOCAL_OUTPUT_DIR.iterdir()):
    dest_path = DRIVE_OUTPUT_DIR / src_path.name
    shutil.copy2(src_path, dest_path)


[setup] Satisfaction LightGBM Optuna study
[setup] input=/content/Stack-Overflow-Survey-Analysis/data/derived/clean_core.parquet
[setup] study=sat_lightgbm_optuna_weighted
[setup] spec=core_with_comp
[setup] rows train=180,317 valid_2024=14,780 test_2025=18,596
[setup] features=36
[setup] objective_metric=weighted_f1 holdout_weight=1.00 rolling_weight=0.00
[setup] tuning seeds=42,52
[setup] device=cpu gpu_backend=n/a gpu_device_id=n/a
[setup] storage=in-memory only
[setup] 2025 test remains untouched during Optuna search
[I 2026-04-12 02:21:55,717] A new study created in memory with name: sat_lightgbm_optuna_weighted
[I 2026-04-12 02:22:35,010] Trial 0 finished with value: 0.38923617773693553 and parameters: {'n_estimators': 400, 'learning_rate': 0.05, 'num_leaves': 63, 'max_depth': -1, 'min_child_samples': 50, 'min_child_weight': 0.001, 'subsample': 0.8, 'subsample_freq': 0, 'colsample_bytree': 0.8, 'reg_alpha': 1e-08, 'reg_lambda': 1e-08, 'min_split_gain': 0.0, 'cat_l2': 10.0, 'cat_s